# Fixed-cap representation: synthetic validation

Step 1 only: mesh extraction, mapping, fixed basis reconstruction and weighted rigid/phase alignment. No boundary optimization, scalar fields, PCA, VAE or mass export.

Run with the existing `organoid-mesh` environment. All reusable numerical and plotting functions are in the package. Outputs are written under `exports/crypt_morphology/`.

The basis combines smooth Fourier extensions of the opening curve with Dirichlet spherical-cap interior harmonics. This shared dictionary permits a free opening; it is **not an orthogonal dictionary as a whole**. Its interior-degree setting controls resolution. The cap pole is a coordinate choice, not a detected tip.

In [ ]:
from pathlib import Path
import sys, os, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src/organograph").is_dir())
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.io as pio
from IPython.display import display
pio.renderers.default = "plotly_mimetype"
from organograph.crypt_morphology import *
from organograph.plotting.crypt_morphology import (
    plot_cap_representation, plot_cap_alignment,
    plot_cap_error_summary, plot_cap_comparison_static,
)
OUT = ROOT / "exports/crypt_morphology"
OUT.mkdir(parents=True, exist_ok=True)
DEGREES = [4, 8, 12]
BASES = {degree: CapBasis.build(degree) for degree in DEGREES}


## Independent analytical shapes

These meshes come from explicit surface formulas, not from the coefficients being fitted. The original triangle interiors supply fitting observations. Bidirectional distances to triangles measure geometric reconstruction accuracy independently of that correspondence loss. Errors are normalized by the original surface RMS radius.

In [ ]:
KINDS = ["hemisphere", "elongated", "constricted", "bent", "asymmetric", "broad_bottom"]
rows, examples = [], {}
for kind in KINDS:
    vertices, faces = synthetic_crypt(kind, n_rings=30, n_sectors=80)
    patch = extract_patch(vertices, faces)
    mapping = parameterize_patch(patch)
    for degree, basis in BASES.items():
        rep = fit_representation(patch, mapping, basis)
        rv, rf = reconstruct(rep, basis, original_coordinates=True, n_rings=40, n_sectors=128)
        metrics = surface_distance_metrics(vertices, faces, rv, rf)
        rows.append(dict(kind=kind, degree=degree, **rep.diagnostics, **metrics))
        examples[kind, degree] = patch, mapping, rep
synthetic_results = pd.DataFrame(rows)
synthetic_results.to_csv(OUT / "synthetic_validation.csv", index=False)
display(synthetic_results[["kind", "degree", "n_modes", "relative_symmetric_rmse",
                           "condition_number", "area_log_ratio_rms"]])
figure = plot_cap_error_summary(synthetic_results)
figure.savefig(OUT / "synthetic_errors.png", dpi=160)
display(figure)
plt.close(figure)

## Inspect the opening and reconstruction

Change `KIND` and `DEGREE`. The cap panel shows the chart, not a replacement of the actual geometry by a sphere. The residual panel compares corresponding source vertices; the table above instead reports point-to-triangle errors.

In [ ]:
KIND, DEGREE = "constricted", 12
patch, mapping, rep = examples[KIND, DEGREE]
display(plot_cap_representation(patch, mapping, rep, BASES[DEGREE]))
figure = plot_cap_comparison_static(patch, rep, BASES[DEGREE])
figure.savefig(OUT / "synthetic_constricted.png", dpi=160)
display(figure)
plt.close(figure)

## Physical size and shape-only representations

`unit_rms` divides by the exact area-weighted RMS radius of the original triangle surface. Surface area and the original RMS radius remain metadata, and the pose restores original units. Switching modes does not refit the chart or coefficients.

In [ ]:
physical = examples["asymmetric", 12][2]
shape_only = with_size_mode(physical, "unit_rms")
basis = BASES[12]
q, _ = basis.reference_mesh()
a = reconstruct(physical, basis, q, original_coordinates=True)
b = reconstruct(shape_only, basis, q, original_coordinates=True)
assert np.allclose(a, b)
print("Original RMS radius:", physical.rms_radius)
print("Physical and normalized reconstructions restore identically:", np.max(np.abs(a-b)))

## Rigid and angular-phase alignment

Known physical rotations/translations and a change in cap angular origin must not create a shape difference. Reflections are prohibited. Real symmetric shapes may have ambiguous phase, which is reported rather than interpreted as a biological orientation.

In [ ]:
from dataclasses import replace
from scipy.spatial.transform import Rotation
reference = physical
rotation = Rotation.from_rotvec([0.4, -0.7, 0.2]).as_matrix()
c = basis.phase_shift(reference.coefficients, 0.413) @ rotation.T
c[0] += [3, -2, 5]
rotated = replace(reference, coefficients=c, pose=SimilarityTransform())
aligned = align_representations([reference, rotated], basis)
max_error = np.max(np.linalg.norm(aligned.aligned[0] - aligned.aligned[1], axis=1))
assert max_error < 1e-5
print("Maximum discrepancy after alignment:", max_error)
print("Converged:", aligned.converged, "phase ambiguous:", aligned.phase_ambiguous)
display(plot_cap_alignment(aligned, basis))
frozen = make_alignment_reference(aligned, basis)
new_result = align_to_reference([rotated], basis, frozen)
assert np.array_equal(new_result.template, frozen.vertices)

## Interchange and limitations

Save the exact basis and the frozen alignment template together with representations. These are representation coefficients, not learned latent vectors. `save_representation(..., patch=..., mapping=...)` also preserves the source correspondence for later marker transfer.

The tests in `tests/crypt_morphology/` additionally cover known transforms, mesh relabeling, area moments under subdivision, mesh-density changes, size scaling, invalid topology and frozen-reference reuse. Arbitrary coefficient vectors are not guaranteed to be embedded valid surfaces. The diagnostics check degenerate triangles, but do not certify absence of self-intersections.